# Group 2 CPECOG2: Coral bleaching, Run 1
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 10 to 15 min).
Outputs: dataset stats, duplicate count, saved split (`splits.csv`), one YOLOv8s-cls training run, validation macro F1 / bleached recall / confusion matrix, model size, CPU latency, and a summary for the progress log. The test set is **not** touched (it is evaluated once at the end of the project).

In [1]:
!pip -q install ultralytics imagehash kagglehub

In [2]:
import os, glob, shutil, time, json, random
import numpy as np, pandas as pd
from PIL import Image
import imagehash
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_SRC = None  # set a folder path here to skip the Kaggle download

## 1. Download the dataset

In [3]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
print(DATA_SRC)
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
paths = [p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts)]
def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
print("Unlabeled files skipped:", df["label"].isna().sum())
df = df.dropna().reset_index(drop=True)
print(df["label"].value_counts())

100%|██████████| 25.8M/25.8M [00:00<00:00, 65.6MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/vencerlanz09/healthy-and-bleached-corals-image-classification/versions/1
Unlabeled files skipped: 0
label
bleached    485
healthy     438
Name: count, dtype: int64


## 2. Statistical analysis (EDA)

In [4]:
rows = []
for p in df["path"]:
    try:
        im = Image.open(p); w, h = im.size
        arr = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255
        rows.append((w, h, *arr.reshape(-1, 3).mean(0), str(imagehash.phash(im.convert("RGB"))), True))
    except Exception:
        rows.append((None, None, None, None, None, None, False))
df[["w", "h", "r", "g", "b", "phash", "ok"]] = pd.DataFrame(rows)
print("Unreadable files:", (~df["ok"]).sum())
df = df[df["ok"]].reset_index(drop=True)
df["aspect"] = df["w"] / df["h"]
print("\nClass counts and ratio:"); print(df["label"].value_counts(normalize=True).round(3))
print("\nImage size:"); print(df[["w", "h", "aspect"]].describe().round(2))
print("\nMean RGB per class:"); print(df.groupby("label")[["r", "g", "b"]].mean().round(3))

Unreadable files: 0

Class counts and ratio:
label
bleached    0.525
healthy     0.475
Name: proportion, dtype: float64

Image size:
             w        h  aspect
count   923.00   923.00  923.00
mean    295.25   231.14    1.31
std      67.32    57.04    0.25
min     150.00   134.00    0.50
25%     300.00   200.00    1.33
50%     300.00   225.00    1.33
75%     300.00   225.00    1.50
max    1024.00  1024.00    2.24

Mean RGB per class:
              r      g      b
label                        
bleached  0.336  0.423  0.405
healthy   0.282  0.406  0.434


## 3. Remove duplicates (perceptual hash), then stratified 70/15/15 split

In [5]:
before = len(df)
dup_mask = df.duplicated("phash", keep="first")
conflict = df[df["phash"].isin(df.loc[dup_mask, "phash"])].groupby("phash")["label"].nunique()
print("Exact perceptual-hash duplicates removed:", int(dup_mask.sum()))
print("Duplicate groups with conflicting labels:", int((conflict > 1).sum()))
df = df[~dup_mask].reset_index(drop=True)
print("Images after de-duplication:", len(df), "of", before)

from sklearn.model_selection import train_test_split
tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
df["split"] = ""
df.loc[tr.index, "split"] = "train"; df.loc[va.index, "split"] = "val"; df.loc[te.index, "split"] = "test"
print(pd.crosstab(df["split"], df["label"], margins=True))
df[["path", "label", "split", "phash"]].assign(path=lambda d: d["path"].map(os.path.basename)).to_csv("splits.csv", index=False)

# Copy into the folder layout Ultralytics expects, padded to square so no coral is cropped out
ROOT = "/content/coral_data" if os.path.isdir("/content") else os.path.abspath("coral_data")
shutil.rmtree(ROOT, ignore_errors=True)
for _, r in df.iterrows():
    d = os.path.join(ROOT, r["split"], r["label"]); os.makedirs(d, exist_ok=True)
    im = Image.open(r["path"]).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    sq.save(os.path.join(d, os.path.splitext(os.path.basename(r["path"]))[0] + ".jpg"), quality=95)
print("Data written to", ROOT)

Exact perceptual-hash duplicates removed: 21
Duplicate groups with conflicting labels: 9
Images after de-duplication: 902 of 923
label  bleached  healthy  All
split                        
test         72       64  136
train       336      295  631
val          72       63  135
All         480      422  902
Data written to /content/coral_data


## 4. Run 1: YOLOv8s-cls, default settings

In [6]:
from ultralytics import YOLO
import torch
EPOCHS = 15
model = YOLO("yolov8s-cls.pt")
t0 = time.time()
model.train(data=ROOT, epochs=EPOCHS, imgsz=224, batch=32, seed=SEED, deterministic=True,
            patience=8, project="runs", name="run1", exist_ok=True,
            device=0 if torch.cuda.is_available() else "cpu")
train_min = (time.time() - t0) / 60
RUN_DIR = str(model.trainer.save_dir)
print("Run folder:", RUN_DIR)
print(f"Training time: {train_min:.1f} min")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.211 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/coral_data, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None

## 5. Validation metrics (macro F1, bleached recall, confusion matrix)

In [7]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, confusion_matrix, classification_report
best = os.path.join(RUN_DIR, "weights", "best.pt")
m = YOLO(best)
val_files = sorted(glob.glob(os.path.join(ROOT, "val", "*", "*.jpg")))
y_true = [os.path.basename(os.path.dirname(p)) for p in val_files]
y_pred = []
for i in range(0, len(val_files), 64):
    for r in m.predict(val_files[i:i+64], imgsz=224, verbose=False):
        y_pred.append(r.names[r.probs.top1])
labels = ["healthy", "bleached"]
macro_f1 = f1_score(y_true, y_pred, average="macro")
bl_recall = recall_score(y_true, y_pred, pos_label="bleached")
acc = accuracy_score(y_true, y_pred)
print(classification_report(y_true, y_pred, labels=labels, digits=3))
print("Confusion matrix (rows = true, cols = predicted):", labels)
print(confusion_matrix(y_true, y_pred, labels=labels))

              precision    recall  f1-score   support

     healthy      0.833     0.873     0.853        63
    bleached      0.884     0.847     0.865        72

    accuracy                          0.859       135
   macro avg      0.859     0.860     0.859       135
weighted avg      0.860     0.859     0.859       135

Confusion matrix (rows = true, cols = predicted): ['healthy', 'bleached']
[[55  8]
 [11 61]]


## 6. Model size and CPU latency

In [8]:
size_mb = os.path.getsize(best) / 1e6
mc = YOLO(best)
imgs = val_files[:100]
for p in imgs[:10]: mc.predict(p, imgsz=224, device="cpu", verbose=False)
t0 = time.time()
for p in imgs: mc.predict(p, imgsz=224, device="cpu", verbose=False)
lat_ms = (time.time() - t0) / len(imgs) * 1000
print(f"best.pt size: {size_mb:.1f} MB | CPU latency: {lat_ms:.1f} ms/image (Colab CPU, batch 1)")

best.pt size: 10.3 MB | CPU latency: 27.9 ms/image (Colab CPU, batch 1)


## 7. Summary for the progress log

In [9]:
counts = pd.crosstab(df["split"], df["label"])
summary = (f"Run 1 ({time.strftime('%b. %d, %Y')}): YOLOv8s-cls, ImageNet weights, default Ultralytics settings, "
           f"{EPOCHS} epochs max, imgsz 224, batch 32, seed {SEED}. "
           f"Data: {len(df)} images after removing {int(dup_mask.sum())} duplicates; "
           f"train/val/test = {(df.split=='train').sum()}/{(df.split=='val').sum()}/{(df.split=='test').sum()}. "
           f"Validation: accuracy {acc:.3f}, macro F1 {macro_f1:.3f}, bleached recall {bl_recall:.3f}. "
           f"best.pt {size_mb:.1f} MB, CPU {lat_ms:.0f} ms/image. Training time {train_min:.1f} min.")
print(summary)
open("run1_summary.txt", "w").write(summary)

Run 1 (Nov. 13, 2026): YOLOv8s-cls, ImageNet weights, default Ultralytics settings, 15 epochs max, imgsz 224, batch 32, seed 42. Data: 902 images after removing 21 duplicates; train/val/test = 631/135/136. Validation: accuracy 0.859, macro F1 0.859, bleached recall 0.847. best.pt 10.3 MB, CPU 28 ms/image. Training time 1.4 min.


329

Download `splits.csv`, `run1_summary.txt`, and from the run folder printed in step 4: `results.csv` and `confusion_matrix.png`, from the Files panel and put them in the shared Drive or repo.